# 01. Sandbox Notebook

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import sys
sys.path.append("../")

# from PlanetaryCA import PlanetaryCA

In [ ]:
import pygame, os, json
from torch.cuda import empty_cache, reset_max_memory_allocated
from importlib.resources import files

from pyca.interface import Camera
from pyca.automata.models import *
from pyca.interface import launch_video, add_frame, print_screen
from pyca.interface.text import TextBlock, DropdownMenu, InputField, render_text_blocks

## 01.A. First CA

In [ ]:
from pyca import Automaton
from matplotlib.colors import hsv_to_rgb, to_rgb, to_rgba
import networkx as nx
from sklearn.metrics.pairwise import euclidean_distances

from joblib import Parallel, delayed

class PlanetaryCA(Automaton):
    def __init__(self, size,
                 n_states: int = 2,
                 n_active: int = 4,
                 dt: float = 1,
                 d_thresh: float = 1,
                 r0: float = 2,
                 r_factor: float = 1,
                 h0: float = 1,
                 h1: float = 1,
                 force_matrix: torch.Tensor = None,
                 seed: float = None,
                 bbox_scale: float = 2,
                 ca_time: int = 100,
                 bouncing: bool = False,
                 viscosity: float = 0.99,
                 rand_v: float = 0.01,
                 mass_max: int = 10,
                 growth: float = 1.0,
                 k_attract: float = 1.0,
                 k_repel: float = 1.0,
                 wind: float = 10,
                 el_size: int = 300,
                 device: str = 'cpu'):
        super().__init__(size)
        
        self.device=device

        if seed is not None:
            torch.manual_seed(seed)

        # Plotting parameters
        # self.el_size = 200
        self.el_size = el_size
        # self.scale = scale
        self.radius = 2

        self.box_size = torch.tensor([bbox_scale*self.h, bbox_scale*self.w], dtype=torch.float32)
        self.scale = self.box_size/bbox_scale
        print(self.scale)

        # CA Parameters
        self.n_states = n_states
        self.n_active = n_active
        self.state_t = torch.zeros(self.h * self.w, dtype=torch.float32)
        self.mass_max = mass_max
        
        # Kinetics
        self.d_thresh = d_thresh
        self.r0 = r0
        self.r_factor = r_factor
        self.h0 = h0
        self.h1 = h1
        self.viscosity = viscosity
        self.rand_v = rand_v
        self.ca_time = ca_time
        self.dt = dt
        self.bouncing = bouncing
        self.growth = torch.tensor(growth)

        # Force matrix for interactions
        if force_matrix is None:
            self.force_matrix = torch.ones((n_states+1, n_states+1), dtype=torch.float32, device=self.device)
        else:
            self.force_matrix = force_matrix

        self.wind = wind
        self._new_world()
        
        self._new_kinetics()

        self._gen_colors()

        self.parallel = Parallel(n_jobs=10, backend="threading")


    def _new_kinetics(self):
        """
        Generates initial kinetics (initialization).
        """
        # self.positions = torch.stack(torch.meshgrid(torch.arange(self.h),torch.arange(self.w),indexing='ij'),
        #                              dim=-1).to(self.device).float() # (H,W,2), positions[x,y] = (x,y)
        # self.positions += torch.tensor([self.h/2+0.5, self.w/2+0.5])
        # self.positions = torch.rand(size=(self.h, self.w, 2))*self.scale + self.scale + torch.tensor([self.h/2, self.w/2], dtype=torch.float32, device=self.device)
        self.positions = torch.rand(size=(self.h, self.w, 2)) * self.box_size
        self.pos_vec = self.positions.view((-1, 2)).float()

        self.v = torch.zeros((self.h * self.w, 2), dtype=torch.float32) # (H * W, 2), velocity of each cell
        # self.m = 0.5*torch.ones(self.h * self.w, dtype=torch.float32) # (H * W), mass of each cell
        self.m = torch.ones(self.h * self.w, dtype=torch.float32) # (H * W), mass of each cell
        self.m[self.active_cells] *= self.mass_max
        # self.m = torch.ones(self.h * self.w, dtype=torch.float32) * self.mass_max # (H * W), mass of each cell

        self.time = 0


    def _new_world(self):
        """
        Generates a new world (initialization).
        """
        # self.active_cells = torch.multinomial(torch.ones(self.h*self.w), self.n_active*self.n_states, replacement=False)
        self.active_cells = torch.arange(self.h*self.w, dtype=torch.int)[self.h*self.w//(self.n_states+1):]
        self.world = torch.zeros((self.h * self.w, self.n_states), dtype=torch.int, device=self.device)
        
        for i in range(self.n_states):
            self.world[self.active_cells[i::self.n_states], i] = 1

        self.one_state = (self.world * torch.arange(1, self.n_states+1).view(1, -1)).sum(dim=1)  # (H * W, n_states)

        self.graph = nx.grid_2d_graph(self.h, self.w) # Create a grid graph for the automaton

        self._new_kinetics()
        self._compute_forces()
        
        self.dirac = torch.zeros(self.n_states)
        self.dirac[0] = 1


    def _gen_colors(self, cmap="coolwarm"):
        """
        Generates a colormap for the automaton.
        """

        cmap = plt.get_cmap("coolwarm", 3)
        white_col = 255*torch.tensor(to_rgba(cmap(1)[:3], alpha=0.5))
        
        if self.n_states == 2:
            cmap = plt.get_cmap(cmap, self.n_states + 1)
            coldic = {0: white_col,
                      1: 255*torch.tensor(to_rgba(cmap(0)[:3], alpha=0.5)),
                      2: 255*torch.tensor(to_rgba(cmap(2)[:3], alpha=0.5))}
        elif self.n_states != 2:
            cmap = plt.get_cmap("turbo", self.n_states + 3)
            coldic = {0: white_col}
            coldic.update({i: 255*torch.tensor(to_rgba(cmap(i+1)[:3], alpha=0.5)) for i in range(1, self.n_states+1)})
        else:
            return NotImplementedError('Colors not implemented for n_states other than 2')
        self.color_dict = coldic


    def _get_neighbors(self):
        neighbor_mask = (self.dist < self.d_thresh) & (self.dist > 0)  # exclude self
        neighbor_mask.fill_diagonal_(False)

        neighbors = [np.where(neighbor_mask[i])[0].tolist() for i in range(len(self.pos_vec))]
        return neighbors
    

    def _get_force_signs(self):

        # Outer comparison matrix
        Si = self.one_state[:, None]
        Sj = self.one_state[None, :]

        # to_zero = (Si > 0) & (Sj == 0)
        to_zero = (Sj > 0) & (Si == 0)

        a_mask = (Si == Sj) * (Si * Sj > 0)
        a_mask = torch.logical_and(a_mask, to_zero)
        a_mask.fill_diagonal_(False)  # Exclude self-comparison

        r_mask = (Si != Sj)
        r_mask.fill_diagonal_(False)  # Exclude self-comparison

        return a_mask, r_mask
    
    
    def _get_force_weights(self):
    
        return self.force_matrix[self.one_state][:, self.one_state]
    

    def _linear_force(self, r0: int=2, h0: int=1, r_factor: int=1, h1: int=1):

        f_out = torch.zeros_like(self.dist)

        sub_r0 = self.dist < r0

        if sub_r0.sum() > 0:
            f_out[sub_r0] = (self.dist[sub_r0] - r0) * (h0 / r0)
        f_out[self.dist >= r0] = (self.dist[self.dist >= r0] - r0) * (h1 / r0)
        f_out[self.dist >= (1+r_factor)*r0] = -(self.dist[self.dist >= (1+r_factor)*r0] - (1+2*r_factor)*r0) * (h1 / r0)
        f_out[self.dist >= (1+2*r_factor)*r0] = 0

        return f_out


    def _compute_forces(self, softening=1e-2, **kwargs):
        delta = self.pos_vec[:, np.newaxis, :] - self.pos_vec[np.newaxis, :, :]  # (N, N, 2)

        # Minimum image convention for periodic boundaries
        if not self.bouncing:
            delta = (delta + self.box_size / 2) % self.box_size - self.box_size / 2

        self.dist = torch.norm(delta, dim=2) + softening # (N, N)
        self.dist.fill_diagonal_(1.0)
        
        # m_product = self.m[:, np.newaxis] * self.m[np.newaxis, :]

        # TODO: Threshold forces for distances above a threshold
        unit_vect = delta / self.dist[:, :, np.newaxis]
        unit_vect[self.dist == 0] = 0

        # attract_mask, repel_mask = self._get_force_signs()

        # # Ensure repel mask includes distances below r0
        # repel_mask = torch.logical_or(repel_mask, self.dist < self.r0)

        # TODO: Define force matrix to assign force sign and strength in a nicer manner
        # - Nutriment cells that are 0, they avoid everyone 
        # - State cells that are 1, 2, ... that avoid each other and are attracted to nutriments 
        # - Dead cells that are only attracting each other (f=0 for the rest) and that become nutriment after some time
        # -- How to add some CA aspects to it ?

        # Initialize force strengths
        F_strength = torch.zeros_like(self.dist)
        F_weight = self._get_force_weights()

        # F_weight[self.dist < self.r0] *= torch.sign(F_weight[self.dist < self.r0])
        F_weight[self.dist < self.r0] = -1

        F_strength = F_weight * self._linear_force(**kwargs)# / self.dist

        # Apply conditional force rules
        # F_strength[attract_mask] = -self.k_attract * m_product[attract_mask] / self.dist[attract_mask]
        # F_strength[repel_mask] = self.k_repel * m_product[repel_mask] / self.dist[repel_mask]

        F_vec = F_strength[:, :, np.newaxis] * unit_vect
        # return F_strength, F_vec, delta
        total_force = torch.sum(F_vec, axis=1)
        return total_force
    

    def _compute_forces_lj(self, epsilon=1.0, sigma=1.0, r_cutoff=5.0, max_force=10, softening=1e-2):
        """
        Computes Lennard-Jones forces between 2D particles.

        Parameters:
        - positions: (N, 2) array of 2D coordinates
        - epsilon: depth of the potential well
        - sigma: characteristic distance
        - r_cutoff: cutoff distance for performance (optional)

        Returns:
        - forces: (N, 2) array of resulting forces on each particle
        """
        forces = np.zeros_like(self.pos_vec)

        # Pairwise displacements
        delta = self.pos_vec[:, np.newaxis, :] - self.pos_vec[np.newaxis, :, :]  # (N, N, 2)

        # Minimum image convention for periodic boundaries
        if not self.bouncing:
            delta = (delta + self.box_size / 2) % self.box_size - self.box_size / 2

        # self.dist = np.linalg.norm(delta, axis=2) + softening
        self.dist = torch.norm(delta, dim=2) + softening # (N, N)
        self.dist.fill_diagonal_(1.0)

        # Apply cutoff
        mask = (self.dist > 0) & (self.dist < r_cutoff)

        # Compute pairwise Lennard-Jones force magnitudes
        inv_r6 = (sigma / self.dist[mask]) ** 6
        inv_r12 = inv_r6 ** 2
        force_mag = 24 * epsilon * (2 * inv_r12 - inv_r6) / self.dist[mask]  # shape (M,)

        # Normalize displacement vectors
        delta_unit = delta[mask] / self.dist[mask][:, None]  # (M, 2)

        # Apply force directions
        # force_vectors = force_mag[:, None] * delta_unit  # (M, 2)
        force_vectors = np.clip(force_mag, -max_force, +max_force)[:, None] * delta_unit  # (M, 2)
        force_vectors[(self.dist < self.r0).view(-1)] *= 1

        # Build full force array by scattering
        i_indices, j_indices = np.where(mask)
        np.add.at(forces, i_indices, -force_vectors)  # F_ij = -F_ji
        np.add.at(forces, j_indices, +force_vectors)

        return forces


    def kinetic_step(self, softening=1e-2):
        forces = self._compute_forces(r_factor=self.r_factor, r0=self.r0, h0=self.h0, softening=softening)

        # forces = self._compute_forces_lj(sigma=2, max_force=1000, r_cutoff=1e5, softening=softening)
        acceleration = forces / (self.m[:, np.newaxis] + softening)
        self.v += acceleration * self.dt
        # add damping
        self.v *= self.viscosity

        # Add drift to speed !
        self.v = self.v * (1-self.rand_v) + torch.randn_like(self.v) * self.rand_v
        

        self.pos_vec += self.v * self.dt

        if self.bouncing:
            # Bouncing walls
            for dim in [0, 1]:  # x and y
                mask_low = self.pos_vec[:, dim] < 0
                mask_high = self.pos_vec[:, dim] > self.box_size[dim]

                # Reflect position
                self.pos_vec[mask_low, dim] = 0
                self.pos_vec[mask_high, dim] = 2*self.box_size[dim] - self.pos_vec[mask_high, dim]

                # Reverse velocity on bounce
                self.v[mask_low | mask_high, dim] *= -1
        else:
            # Wrap positions back into the box
            self.pos_vec = self.pos_vec % self.box_size

        self.positions = self.pos_vec.view((self.w, self.h, 2))

        self.time += 1
    

    def ca_step(self):
        new_world = self.world.detach().clone() # (H * W, 2), copy of the world
        # new_world = torch.zeros_like(self.world) # (H * W, 2), copy of the world
        neighbors = self._get_neighbors()

        for n_i, neigh in enumerate(neighbors):
            neigh_count = self.world[neigh].sum(axis=0)
            max_state = neigh_count.argmax().item()

            # Decay if a cell is alone
            if len(neigh) == 0 and self.world[n_i].sum() != 0:
                self.m[n_i] -= self.growth

            # State 0 can be "converted"
            if neigh_count.sum() != 0 and self.world[n_i].sum() == 0:
                new_world[n_i] = torch.roll(self.dirac, max_state)
                self.m[n_i] = self.mass_max

            # if torch.count_nonzero(neigh_count) == 1 and neigh_count[max_state] > 3:
            #     self.m[n_i] -= 1*self.growth

            # if neigh_count[max_state] > 3:
            #     new_world[n_i] = 0
            #     self.m[n_i] = 1

        # Reset cells with small mass
        new_world[self.m <= 0, :] = 0 
        self.m[self.m <= 0] = 1

        # self.state_t += ((new_world * torch.arange(1, self.n_states+1)).sum(dim=1) == (self.world * torch.arange(1, self.n_states+1)).sum(dim=1)).int()
        # new_world[self.state_t > 10] = 0 # Reset cells which have not changed for a long time
        # self.m = - (new_world * torch.arange(1, self.n_states+1)).sum(dim=1) + (self.n_states+1)

        return new_world


    def step(self):
        """
            Makes one step of the automaton
        """
        self.kinetic_step()

        if self.time > self.ca_time:
            self.time = 0
            
            self.world = self.ca_step()
        
        self.one_state = (self.world * torch.arange(1, self.n_states+1).view(1, -1)).sum(dim=1)  # (H * W, n_states)

    def step_old(self):
        """
            Makes one step of the automaton
        """
        new_world = torch.zeros_like(self.world) # (H,W,3), copy of the world

        # n_l = self.world.roll(-1, dims=0)
        # n_r = self.world.roll(1, dims=0)
        
        # n_a = self.world.roll(-1, dims=1)
        # n_b = self.world.roll(1, dims=1)
        
        for y in range(self.h):
            for x in range(self.w) : # loop over all cells
                sum_neigh1 = 0 # initialize the neighbour sum
                sum_neigh2 = 0
                for dx,dy in [(1,1),(1,0),(1,-1),(0,1),(0,-1),(-1,-1),(-1,0),(-1,1)]: # Iterate over Neighbors
                # for dx,dy in [(1,0),(0,1),(0,-1),(-1,0)]: # Iterate over Neighbors (4)
                    sum_neigh1 += self.world[(y+dy)%self.h,(x+dx)%self.w, 0] == 1 # State 1
                    sum_neigh2 += self.world[(y+dy)%self.h,(x+dx)%self.w, 1] == 1 # State 2
                if self.world[y,x].sum() == 0 : # Current cell is dead
                    if(sum_neigh1 > sum_neigh2): # Condition for it to become S1
                        new_world[y,x, 0] = 1
                    elif(sum_neigh1 < sum_neigh2): # Condition for it to stay S0
                        new_world[y,x, 1] = 1
                elif self.world[y,x, 0] == 1: # Current cell is S1
                    if(sum_neigh2 > sum_neigh1 + 1):
                        new_world[y,x, 1] = 1
                    else:
                        new_world[y,x, 0] = 1
                else:
                    if(sum_neigh1 > sum_neigh2 + 1):
                        new_world[y,x, 0] = 1
                    else:
                        new_world[y,x, 1] = 1

        self.time+=1

        self.world = new_world


    def reset(self):
        """
            Resets the automaton to the initial state.
        """
        self._worldmap = torch.zeros_like(self._worldmap)
        self.time=0

        self._new_world()

    
    def process_event(self, event, camera=None):
        """
        DEL -> resets the automaton
        UP -> increase the number of starting cells by 1 (max 20)
        DOWN -> decrease the number of starting cells by 1 (min 2)
        """
        if(event.type == pygame.KEYDOWN):
            if event.key == pygame.K_BACKSPACE or event.key == pygame.K_DELETE:
                self.reset() 
                self.draw()

            if event.key == pygame.K_UP:
                self.n_active = torch.min(torch.tensor([self.n_active+1, 20]))
            if event.key == pygame.K_DOWN:
                self.n_active = torch.max(torch.tensor([self.n_active-1, 2]))


    def draw(self):
        """
            Draws the current state of the automaton, using arbitrary coloration which looks nice.
        """
        # self._worldmap=self.get_color_world() # (3,H,W)
        # self._worldmap = self.world.view(-1, self.n_states)
        # self._worldmap = self.world * torch.arange(1, self.n_states+1)
        pass


    @property
    def worldsurface(self):
        surf = pygame.Surface((self.el_size*self.w, self.el_size*self.h))
        # surf = pygame.Surface((self.scale[0]*self.w, self.scale[1]*self.h))
        surf.fill("black")
        
        # x_pos, y_pos = (self.positions.view(-1, 2) * self.scale).t().int()
        x_pos, y_pos = (self.pos_vec * self.scale).t().int()

        # x_pos, y_pos = torch.zeros_like(self.positions.view(2, -1), dtype=torch.int32)

        # n_wrap_dist = torch.norm(self.pos_vec[:, None, :] - self.pos_vec[None, :, :], dim=2)
        # a_mat = (n_wrap_dist < self.d_thresh).int().numpy()
        # np.fill_diagonal(a_mat, 0)

        # edges = list(nx.Graph(a_mat).edges())
        
        # for e in edges:
        #     # pygame.draw.line(surface, color, start_pos, end_pos, width=1)
        #     pygame.draw.line(surf, "gray", (x_pos[e[0]], y_pos[e[0]]), (x_pos[e[1]], y_pos[e[1]]), width=1)

        for x, y, state, mass in zip(x_pos, y_pos, self.one_state, self.m):
            pygame.draw.circle(surf, self.color_dict[state.item()].numpy(), (x, y), self.radius+mass.int())
        
        return surf
 
    # @property
    # def worldmap(self):
    #     return self._worldmap.view(-1, self.n_states)
 

In [ ]:
planca = PlanetaryCA((3, 3), n_states=2, #seed=240896,
                     ca_time=1, r0=1,
                     n_active=1, d_thresh=2, device='cpu')



In [ ]:
def lennard_force(r, sigma=1):
    return 48*(sigma**12 /r**13 - 0.5 * sigma**6 / r**7)


x = torch.linspace(0.1, 5, 1000)

for sigma in np.arange(0.5, 2.5, 0.5):
    plt.plot(x, lennard_force(x, sigma), label=f'sigma={sigma}')

plt.ylim(-10, 10)
plt.legend()

## Run in notebook ?

In [ ]:
# === Pygame Visualization ===

def run_simulation(screen_size=(800, 800), graph_size=(6, 6), n_states=2, fps=60, n_steps=None, **kwargs):
    pygame.init()
    screen = pygame.display.set_mode(screen_size)

    pygame.display.set_caption("Planetary Cellular Automata Simulation")
    programIcon = pygame.image.load(files('pyca.interface.files').joinpath('icon.png'))
    pygame.display.set_icon(programIcon)
    clock = pygame.time.Clock()

    planca = PlanetaryCA(graph_size, n_states=n_states, n_active=4, device='cpu', **kwargs)
    
    scale = torch.tensor(screen_size)/planca.box_size

    planca.scale = scale

    running = True
    stopped = True

    if n_steps is not None:
        stopped = False
    
    step_count = 0

    while running:
        screen.fill((10, 10, 20))
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.KEYDOWN :
                if(event.key == pygame.K_SPACE): # Press 'SPACE' to start/stop the automaton
                    stopped=not(stopped)
                if(event.key == pygame.K_q):
                    running=False
                if(event.key == pygame.K_s):
                    planca.step()

            planca.process_event(event) # Process the event in the automaton

        if(not stopped):
            planca.step()

        planca.draw()
        
        world_surface = planca.worldsurface

        screen.blit(world_surface, (1,1))

        pygame.display.flip()
        clock.tick(fps)

        if n_steps is not None and not stopped:
            step_count += 1
            if step_count >= n_steps:
                pygame.quit()
                running = False

    pygame.quit()

In [ ]:
force_mat = torch.tensor([[-1, 1, 1],
                          [-1, -1, -0.8],
                          [-1, 0.6, -1]], dtype=torch.float32)

force_mat = torch.tensor([[-1, 1, 1, 1],
                          [-0.9, -1, -0.6, 0.8],
                          [-0.9, 0.6, -1, -0.6],
                          [-0.9, -0.8, 0.6, -1]], dtype=torch.float32)

# Good Particle Simulation
# run_simulation((1200, 800), graph_size=(20, 20), n_states=3, fps=60, dt=0.02,
#                d_thresh=6, ca_time=2, r0=5, h0=10, h1=2, r_factor=4, viscosity=0.98,
#                seed=220367, growth=1, bbox_scale=5, mass_max=4,
#                force_matrix=force_mat)

run_simulation((1200, 800), graph_size=(20, 20), n_states=3, fps=60, dt=0.1,
               d_thresh=5, ca_time=2, r0=3, h0=10, h1=2, r_factor=1, viscosity=0.98,
               seed=220367, growth=1, bbox_scale=5, mass_max=4, el_size=100,
               force_matrix=force_mat)

## Benchmark Perf

In [ ]:
from pyinstrument import Profiler

planca = PlanetaryCA(size=(20, 20), n_states=3, dt=0.02,
               d_thresh=6, ca_time=2, r0=5, h0=10, h1=2, r_factor=4, viscosity=0.98,
               seed=220367, growth=1, bbox_scale=5, mass_max=4,
               force_matrix=force_mat)

profiler = Profiler()
profiler.start()

run_simulation((1200, 800), n_steps=100, graph_size=(40, 40), n_states=3, fps=30, dt=0.02,
               d_thresh=6, ca_time=2, r0=5, h0=10, h1=2, r_factor=3, viscosity=0.98,
               seed=220367, growth=1, bbox_scale=5, mass_max=4,
               force_matrix=force_mat)

profiler.stop()

In [ ]:
profiler.print(show_all=True)
# profiler.open_in_browser()

In [ ]:
profiler.print(show_all=True)
# profiler.open_in_browser()